# 4_1_io_model_comparison_large

2020-2025 大样本模型横向比较入口。

- 保留 `4_io_model_comparison_ioexp.ipynb` 作为小样本快速验证。
- 本 notebook 支持 `speed / balance / accuracy` 三档，并提供窗口级缓存与断点续跑。

In [1]:
import os
import sys
import time
import pickle
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.append('../models')

import ioexp_runner as io_runner_module
from ioexp_baselines import price_bs_flat, price_heston, price_localvol_quadratic, price_sabr
from ioexp_data_contract import IOSliceConfig, generate_rolling_slices, select_slice
from ioexp_eval_metrics import compute_group_metrics
from ioexp_runner import IOExperimentRunner, RunnerConfig


In [2]:
# -----------------------------
# 运行模式与输入路径
# -----------------------------
MODE = 'balance'  # 'speed' | 'balance' | 'accuracy'

RAW_CSV = Path('../data/io_option_cross_section_20200101_20251231.csv')
if not RAW_CSV.exists():
    # 当前仓库若暂无全样本文件，先回退到小样本文件做流程验证
    RAW_CSV = Path('../data/io_option_cross_section_20240101_20240131.csv')

RFSV_PATH = Path('../data/rfsv_predictions_20200101_20251231.csv')
RFSV_ARG = str(RFSV_PATH) if RFSV_PATH.exists() else None

CACHE_ROOT = Path('../data/ioexp_large_cache')
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

print('MODE =', MODE)
print('RAW_CSV =', RAW_CSV)
print('RFSV_ARG =', RFSV_ARG)

MODE = balance
RAW_CSV = ../data/io_option_cross_section_20200101_20251231.csv
RFSV_ARG = None


In [3]:
# -----------------------------
# 可选：自动抓取 2020-2025 IO 期权截面（带库存检查）
# -----------------------------
FORCE_REFETCH = True
RAW_CSV = Path('../data/io_option_cross_section_20200101_20251231.csv')

if RAW_CSV.exists() and not FORCE_REFETCH:
    print(f'Found existing dataset, skip fetching: {RAW_CSV}')
else:
    from OptionDataFetcher import OptionDataFetcher
    from dotenv import load_dotenv
    load_dotenv()

    RAW_CSV.parent.mkdir(parents=True, exist_ok=True)
    fetcher = OptionDataFetcher()
    fetcher.init_connection(os.environ.get('RQDATAC_LICENSE') or os.environ.get('RQ_API_KEY'))

    _ = fetcher.fetch_option_panel_for_underlying(
        underlying_order_book_id='000300.XSHG',
        start_date='20200101',
        end_date='20251231',
        risk_free_rate=0.02,
        dividend_yield=0.0,
        apply_cleaning=True,
        save_path=str(RAW_CSV),
        include_api_iv=False,
        verbose=True,
        min_days_to_expiry=5,
        min_volume=0.0,
        max_abs_log_moneyness=None,
        contract_snapshot_step=10
    )
    print(f'Fetch done. saved to: {RAW_CSV}')

✓ 米筐数据连接成功
[OptionDataFetcher] step1/4 获取时间段合约池
[OptionDataFetcher] step2/4 批量拉取期权价格: 4296 个合约 (snapshot_days=147, step=10)
[OptionDataFetcher] step3/4 拉取标的数据并合并
✓ 已保存期权面板数据: ../data/io_option_cross_section_20200101_20251231.csv
✓ 期权面板完成: 339488 行, 覆盖 1455 个交易日
Fetch done. saved to: ../data/io_option_cross_section_20200101_20251231.csv


In [4]:
# -----------------------------
# 三档配置（速度/平衡/精度）
# -----------------------------
PROFILE = {
    'speed': {
        'data_cfg': IOSliceConfig(train_days=126, valid_days=21, test_days=21, step_days=21, min_total_days=168),
        'mc_stage1_paths': 3000,
        'mc_stage2_paths': 8000,
        'maxiter_stage1': 12,
        'maxiter_stage2': 20,
        'baseline_models': ['bs_flat', 'localvol_quadratic_iv'],
    },
    'balance': {
        'data_cfg': IOSliceConfig(train_days=252, valid_days=42, test_days=42, step_days=21, min_total_days=336),
        'mc_stage1_paths': 6000,
        'mc_stage2_paths': 16000,
        'maxiter_stage1': 20,
        'maxiter_stage2': 35,
        'baseline_models': ['bs_flat', 'localvol_quadratic_iv', 'sabr_hagan'],
    },
    'accuracy': {
        'data_cfg': IOSliceConfig(train_days=252, valid_days=63, test_days=63, step_days=21, min_total_days=420),
        'mc_stage1_paths': 8000,
        'mc_stage2_paths': 24000,
        'maxiter_stage1': 30,
        'maxiter_stage2': 50,
        'baseline_models': ['bs_flat', 'localvol_quadratic_iv', 'sabr_hagan', 'heston_cf'],
    },
}

cfg_p = PROFILE[MODE]
BASELINE_FUNCS = {
    'bs_flat': price_bs_flat,
    'localvol_quadratic_iv': price_localvol_quadratic,
    'sabr_hagan': price_sabr,
    'heston_cf': price_heston,
}

print('selected baselines:', cfg_p['baseline_models'])
print('data_cfg:', cfg_p['data_cfg'])

selected baselines: ['bs_flat', 'localvol_quadratic_iv', 'sabr_hagan']
data_cfg: IOSliceConfig(train_days=252, valid_days=42, test_days=42, step_days=21, min_total_days=336)


In [5]:
# -----------------------------
# 覆盖 runner 内 baseline 调度（仅在本 notebook 会话中生效）
# -----------------------------
def run_selected_baselines(option_df: pd.DataFrame):
    out = {}
    for name in cfg_p['baseline_models']:
        out[name] = BASELINE_FUNCS[name](option_df)
    return out

io_runner_module.run_all_baselines = run_selected_baselines
print('patched ioexp_runner.run_all_baselines ->', cfg_p['baseline_models'])

patched ioexp_runner.run_all_baselines -> ['bs_flat', 'localvol_quadratic_iv', 'sabr_hagan']


In [6]:
# -----------------------------
# 读取数据并构建 runner（按档位覆盖 MC 与校准迭代）
# -----------------------------
raw_df = pd.read_csv(RAW_CSV)

base_runner = IOExperimentRunner()
cfg = replace(
    base_runner.cfg,
    data=cfg_p['data_cfg'],
    mc_stage1=replace(base_runner.cfg.mc_stage1, n_paths=cfg_p['mc_stage1_paths']),
    mc_stage2=replace(base_runner.cfg.mc_stage2, n_paths=cfg_p['mc_stage2_paths']),
    calib=replace(
        base_runner.cfg.calib,
        maxiter_stage1=cfg_p['maxiter_stage1'],
        maxiter_stage2=cfg_p['maxiter_stage2'],
    ),
)
runner = IOExperimentRunner(cfg=cfg)
prepared = runner.prepare_dataset(raw_df)

print('prepared rows:', len(prepared), 'unique trade days:', prepared['trade_date'].nunique())

prepared rows: 293800 unique trade days: 1455


In [7]:
# -----------------------------
# 生成窗口；若数据太短则自动降级为 demo 小窗口
# -----------------------------
windows = generate_rolling_slices(prepared, config=cfg.data)

if len(windows) == 0:
    demo_cfg = IOSliceConfig(train_days=10, valid_days=5, test_days=5, step_days=5, min_total_days=20)
    windows = generate_rolling_slices(prepared, config=demo_cfg)
    active_data_cfg = demo_cfg
    print('No windows under selected mode; fallback to demo cfg:', active_data_cfg)
else:
    active_data_cfg = cfg.data

print('n_windows =', len(windows))
windows[:2] if len(windows) else []

n_windows = 54


[{'train_start': Timestamp('2020-01-02 00:00:00'),
  'train_end': Timestamp('2021-01-14 00:00:00'),
  'valid_start': Timestamp('2021-01-15 00:00:00'),
  'valid_end': Timestamp('2021-03-22 00:00:00'),
  'test_start': Timestamp('2021-03-23 00:00:00'),
  'test_end': Timestamp('2021-05-25 00:00:00')},
 {'train_start': Timestamp('2020-02-10 00:00:00'),
  'train_end': Timestamp('2021-02-19 00:00:00'),
  'valid_start': Timestamp('2021-02-22 00:00:00'),
  'valid_end': Timestamp('2021-04-21 00:00:00'),
  'test_start': Timestamp('2021-04-22 00:00:00'),
  'test_end': Timestamp('2021-06-24 00:00:00')}]

In [8]:
# -----------------------------
# 窗口级断点续跑：每个窗口独立缓存
# -----------------------------
cache_dir = CACHE_ROOT / f"{MODE}_windows"
cache_dir.mkdir(parents=True, exist_ok=True)

summary_rows = []
group_rows = []

horizon = active_data_cfg.train_days + active_data_cfg.valid_days + active_data_cfg.test_days
single_cfg = replace(
    cfg,
    data=IOSliceConfig(
        train_days=active_data_cfg.train_days,
        valid_days=active_data_cfg.valid_days,
        test_days=active_data_cfg.test_days,
        step_days=horizon,
        min_total_days=horizon,
    ),
)

t0 = time.time()
for i, win in enumerate(windows, 1):
    test_start = pd.Timestamp(win['test_start']).strftime('%Y%m%d')
    test_end = pd.Timestamp(win['test_end']).strftime('%Y%m%d')
    cache_file = cache_dir / f'window_{i:04d}_{test_start}_{test_end}.pkl'

    if cache_file.exists():
        with open(cache_file, 'rb') as f:
            res = pickle.load(f)
        loaded = True
    else:
        sub_df = select_slice(prepared, pd.Timestamp(win['train_start']), pd.Timestamp(win['test_end']))
        sub_runner = IOExperimentRunner(cfg=single_cfg)
        res = sub_runner.run_single_test_window(
            prepared_df=sub_df,
            rfsv_pred_path=RFSV_ARG,
            H_init=0.1,
            eta_init=1.0,
            rho_init=-0.7,
        )
        with open(cache_file, 'wb') as f:
            pickle.dump(res, f)
        loaded = False

    s = res['summary'].copy()
    s['window_id'] = i
    s['test_start'] = pd.Timestamp(win['test_start'])
    s['test_end'] = pd.Timestamp(win['test_end'])
    s['loaded_from_cache'] = loaded
    summary_rows.append(s)

    rb = res['rbergomi_test_df_with_rfsv_prior']
    if rb is not None and not rb.empty and 'maturity_bucket' in rb.columns:
        gm = compute_group_metrics(rb, 'maturity_bucket')
        gm['window_id'] = i
        gm['test_start'] = pd.Timestamp(win['test_start'])
        gm['test_end'] = pd.Timestamp(win['test_end'])
        gm['group_type'] = 'maturity_bucket'
        group_rows.append(gm)

    if rb is not None and not rb.empty and 'moneyness_bucket' in rb.columns:
        gk = compute_group_metrics(rb, 'moneyness_bucket')
        gk['window_id'] = i
        gk['test_start'] = pd.Timestamp(win['test_start'])
        gk['test_end'] = pd.Timestamp(win['test_end'])
        gk['group_type'] = 'moneyness_bucket'
        group_rows.append(gk)

    if i % 5 == 0 or i == len(windows):
        print(f'[{i}/{len(windows)}] processed. elapsed={(time.time()-t0)/60:.2f} min')

elapsed_min = (time.time() - t0) / 60.0
print(f'All windows done. elapsed={elapsed_min:.2f} min')

KeyboardInterrupt: 

In [ ]:
# -----------------------------
# 汇总输出：每窗口明细 + 全局汇总
# -----------------------------
summary_by_window = pd.concat(summary_rows, ignore_index=True) if summary_rows else pd.DataFrame()
group_metrics_all = pd.concat(group_rows, ignore_index=True) if group_rows else pd.DataFrame()

summary_dir = CACHE_ROOT / MODE
summary_dir.mkdir(parents=True, exist_ok=True)

summary_by_window_csv = summary_dir / 'summary_by_window.csv'
summary_aggregate_csv = summary_dir / 'summary_aggregate.csv'
group_metrics_csv = summary_dir / 'group_metrics_all.csv'

if not summary_by_window.empty:
    summary_by_window.to_csv(summary_by_window_csv, index=False)
    summary_aggregate = (
        summary_by_window.groupby('model', as_index=False)
        .agg(
            iv_rmse_mean=('iv_rmse', 'mean'),
            iv_rmse_std=('iv_rmse', 'std'),
            iv_mae_mean=('iv_mae', 'mean'),
            price_rmse_mean=('price_rmse', 'mean'),
            n_windows=('window_id', 'nunique'),
        )
        .sort_values('iv_rmse_mean')
        .reset_index(drop=True)
    )
    summary_aggregate.to_csv(summary_aggregate_csv, index=False)
else:
    summary_aggregate = pd.DataFrame()

if not group_metrics_all.empty:
    group_metrics_all.to_csv(group_metrics_csv, index=False)

print('saved:', summary_by_window_csv)
print('saved:', summary_aggregate_csv)
print('saved:', group_metrics_csv)
summary_aggregate.head(20)

In [ ]:
# -----------------------------
# 可视化1：模型平均 IV RMSE 排名
# -----------------------------
if summary_aggregate.empty:
    print('summary_aggregate is empty.')
else:
    plt.figure(figsize=(8, 4))
    plt.bar(summary_aggregate['model'], summary_aggregate['iv_rmse_mean'])
    plt.title(f'Model Ranking by Mean IV RMSE ({MODE})')
    plt.xlabel('model')
    plt.ylabel('mean iv_rmse')
    plt.xticks(rotation=30)
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# -----------------------------
# 可视化2：按 bucket 的分组误差（示例：rbergomi_with_rfsv_prior）
# -----------------------------
if group_metrics_all.empty:
    print('group_metrics_all is empty.')
else:
    gm_plot = group_metrics_all[group_metrics_all.get('group_type', '') == 'maturity_bucket'].copy()
    gm_plot = gm_plot[gm_plot['group'].notna()]
    if gm_plot.empty:
        print('No maturity_bucket group metrics.')
    else:
        agg = (
            gm_plot.groupby('group', as_index=False)
            .agg(iv_rmse=('iv_rmse', 'mean'))
            .sort_values('group')
        )
        plt.figure(figsize=(7, 4))
        plt.bar(agg['group'], agg['iv_rmse'])
        plt.title(f'Rough Group Error by Maturity Bucket ({MODE})')
        plt.xlabel('maturity_bucket')
        plt.ylabel('mean iv_rmse')
        plt.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        plt.show()
        agg

In [ ]:
# -----------------------------
# speed 模式验收（目标：1小时内）
# -----------------------------
if MODE != 'speed':
    print(f'Current MODE={MODE}; speed SLA check skipped.')
else:
    target_min = 60.0
    print(f'elapsed_min={elapsed_min:.2f}, target_min={target_min:.2f}')
    if elapsed_min <= target_min:
        print('SPEED MODE SLA: PASS (<= 1 hour)')
    else:
        print('SPEED MODE SLA: NOT PASS (> 1 hour)')

    # 粗略估计：若只跑了部分窗口，可按均值外推
    n_done = len(summary_rows)
    n_total = len(windows)
    if n_done > 0 and n_total > 0:
        est_full = elapsed_min * (n_total / n_done)
        print(f'Estimated full-run time: {est_full:.2f} min for {n_total} windows')